# Lab 5 Task 4: Real N310 OFDM Link

Verify file sizes and finite-transmission accounting, acquire uncropped files with the supplied GNU Radio graphs, and run the same receiver used in Task 3 on data captured on the N310


In [ ]:
from pathlib import Path
import json
import os
import sys
import numpy as np
from IPython.display import Image, display
import IPython.core.pylabtools as pylabtools

# Matplotlib 3.6 expects this compatibility table, which newer IPython omits.
if not hasattr(pylabtools, "backend2gui"):
    pylabtools.backend2gui = {}

def locate_lab_root():
    for base in (Path.cwd(), *Path.cwd().parents):
        if (base / "support" / "lab5_dsp.py").is_file():
            return base
    raise FileNotFoundError("Open this notebook from its Lab 5 folder.")

LAB_ROOT = locate_lab_root()
sys.path.insert(0, str(LAB_ROOT / "support"))
WORK_DIR = LAB_ROOT / "work"
FIGURE_DIR = WORK_DIR / "figures"
WAVEFORM_DIR = WORK_DIR / "waveforms"
CAPTURE_DIR = Path(os.environ.get("ECE7377_LAB5_CAPTURE_DIR", WORK_DIR / "captures"))
for path in (FIGURE_DIR, WAVEFORM_DIR, CAPTURE_DIR):
    path.mkdir(parents=True, exist_ok=True)

import lab5_dsp as lab
import lab5_tools

print("Lab root:", LAB_ROOT)
print("Capture directory:", CAPTURE_DIR)


## Finite-file and capture calculations


In [ ]:
sample_rate_sps = 1_000_000
period_samples = 8192
repeat_count = 13
warmup_periods = 1
acquisition_lead_samples = 2_000_000
capture_seconds = 15.0

# To do: calculate exact sample and byte counts.
payload_burst_samples = ...
transmit_samples = ...
transmit_bytes = ...
capture_samples = ...
capture_bytes = ...
expected_frames = ...

print("OFDM burst samples:", payload_burst_samples)
print("guarded TX samples and bytes:", transmit_samples, transmit_bytes)
print("RX samples and bytes:", capture_samples, capture_bytes)
print("graded frames:", expected_frames)


In [ ]:
assert payload_burst_samples == 106_496
assert transmit_samples == 2_106_496
assert transmit_bytes == 16_851_968
assert capture_samples == 15_000_000
assert capture_bytes == 120_000_000
assert expected_frames == 12
print("Hardware accounting passed.")


## Build the finite transmitter file and metadata


In [ ]:
# To do: enter the three assigned TX gains and the fixed RX gain.
tx_gains_db = (..., ..., ...)
rx_gain_db = ...

prepared = lab5_tools.prepare_hardware_experiment(
    WAVEFORM_DIR,
    CAPTURE_DIR,
    tx_gains_db=tx_gains_db,
    rx_gain_db=rx_gain_db,
)
print(json.dumps(prepared, indent=2))
assert prepared["waveform_bytes"] == 16_851_968


## GNU Radio acquisition

Open `flowgraphs/lab5_finite_ofdm_receiver.grc` on the RX desktop and `flowgraphs/lab5_finite_ofdm_transmitter.grc` on the TX desktop. Verify the assigned device argument, subdevice, antenna, 1 MS/s rate, 2.45 GHz center, gain, and file paths.

1. With TX stopped, set the RX output to `work/captures/lab5_noise_c64.dat`, set the capture to 0.25 seconds, and run RX once.
2. Restore the RX capture to 15 seconds.
3. For each assigned TX gain, set the RX output to `work/captures/lab5_ofdm_tx<gain>_c64.dat`.
4. Start RX first, then TX. Do not crop the raw file.
5. Confirm that the noise file is 2,000,000 bytes and every gain file is 120,000,000 bytes.


## Analyze the uncropped hardware data


In [ ]:
summary_path = CAPTURE_DIR / "lab5_measurement_summary.json"
required = [summary_path, CAPTURE_DIR / "lab5_noise_c64.dat"]
if not all(path.exists() for path in required):
    raise FileNotFoundError("Capture the noise and three gain files before running this cell.")

hardware = lab5_tools.analyze_hardware(CAPTURE_DIR, FIGURE_DIR)
for gain_name, row in hardware["gain_results"].items():
    print(gain_name, json.dumps(row["aggregate"], indent=2))


In [ ]:
for name in ('task4_hardware_acquisition', 'task4_hardware_constellations', 'task4_hardware_channel_estimates', 'task4_hardware_gain_metrics', 'task4_hardware_subcarrier_error'):
    path = FIGURE_DIR / f"{name}.png"
    display(Image(filename=path))


In [ ]:
baseline_gain = tx_gains_db[len(tx_gains_db)//2]
baseline = hardware["gain_results"][f"tx{baseline_gain}"]["aggregate"]
analyzed_frames = baseline["analyzed_frames"]
expected_frames = 12

# To do: compute the acquisition rate for the baseline condition.
baseline_detection_rate = ...
print("baseline detection rate:", baseline_detection_rate)


## Report work

1. Include the TX and RX flowgraphs and verify every hardware field.
2. Report detected frames, analyzed frames, detection rate, BER, PER, EVM, CFO mean and standard deviation, channel power, and estimated SNR for each gain.
3. Show one frame from coarse acquisition through CRC.
4. Interpret the measured channel magnitude and phase across subcarriers.
5. Identify corrupted packets explicitly. An undetected or failed packet is not BER zero.
6. Explain whether increasing gain improved EVM, CRC success, and channel-power estimates without clipping.
